# Olist Seller Agent



## Conectar o Google Drive

Para rodar no Colab, precisamos montar o Drive para acessar os CSVs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Carregar dados relevantes

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

PASTA = Path('/content/drive/MyDrive/1AIAT/Tech_Challange/FASE1/archive/')



## Analise de reclamações onde identificamos que muita elas estão ligadas a logística.

A média de atraso entre as reclamações é 31%. Acima da linha, é entrega; abaixo, é produto.

69% das reclamações não tiveram atraso nenhum. Zerar o atraso não resolveria a maior parte delas.

Esse número faz duas coisas ao mesmo tempo:

Ele dimensiona o invisível. Defeito, produto errado, qualidade abaixo e entrega parcial não aparecem em nenhum indicador operacional — o pedido consta entregue, no prazo.
Ele prova que um Agente de Reviews não é redundante com um de Logística. São populações majoritariamente disjuntas. Sem esse teste, propor dois agentes seria indefensável.

In [ ]:
# ══════════ G9 — % de atraso por tipo de reclamação ══════════
rec = reclamacoes.merge(
    ent[['order_id', 'atrasado', 'order_delivered_customer_date']],
    on='order_id', how='left')
rec_ent = rec[rec['order_delivered_customer_date'].notna()]

perfil = rec_ent.groupby('tipo').agg(
    n=('review_id', 'size'), pct_atraso=('atrasado', 'mean'))
perfil['pct_atraso'] *= 100
perfil = perfil.sort_values('pct_atraso')
media = rec_ent['atrasado'].mean()*100

fig, ax = plt.subplots(figsize=(10.5, 5.4))
cores = [VERMELHO if v > media else AZUL for v in perfil['pct_atraso']]
ax.barh(range(len(perfil)), perfil['pct_atraso'], color=cores, height=0.68)

for i, (v, n_) in enumerate(zip(perfil['pct_atraso'], perfil['n'])):
    ax.annotate(f'{v:.0f}%   ({n_:,} casos)'.replace(',', '.'), xy=(v, i),
                xytext=(6, 0), textcoords='offset points', va='center',
                fontsize=8.8, color='#4a535f')

ax.axvline(media, color=CINZA, linewidth=1.4, linestyle='--', zorder=4)
ax.annotate(f'média: {media:.0f}%', xy=(media, len(perfil)-0.3), xytext=(6, 0),
            textcoords='offset points', color=CINZA, fontsize=8.5)

ax.set_yticks(range(len(perfil)))
ax.set_yticklabels(perfil.index, fontsize=9.5)
ax.set_title('Quais reclamações são de logística — e quais não são')
ax.set_xlabel('% das reclamações daquele tipo que tiveram atraso')
ax.set_xlim(0, perfil['pct_atraso'].max()*1.55)
ax.grid(axis='y', visible=False)
plt.tight_layout(); plt.show()